# cnn_mel

Trains the log-mel CNN, exports int8 TFLite, writes the Rust sample file.

In [ ]:
from typing import TYPE_CHECKING
from utils import (
    TARGET_FRAMES_MEL,
    get_paths,
    configure_tf_runtime,
    set_global_seed,
)

if TYPE_CHECKING:
    import keras
else:
    from tensorflow import keras

configure_tf_runtime()
set_global_seed()

MODEL_STEM = "cnn_mel_tf"
paths = get_paths(MODEL_STEM)
OUT_TFLITE = paths.out_tflite
OUT_AUDIO_RS = paths.out_audio_rs
BATCH_SIZE = 32



In [ ]:
from utils import (
    make_mel_datasets,
    NUM_MEL_BINS_MEL,
)

train_ds, val_ds, test_ds, label_names = make_mel_datasets(
    num_mel_bins=NUM_MEL_BINS_MEL,
    target_frames=TARGET_FRAMES_MEL,
)
num_labels = len(label_names)
print("Classes:", label_names)



In [ ]:
from utils import NUM_MEL_BINS_MEL, TARGET_FRAMES_MEL, init_wandb, get_callbacks, finish_wandb

CONV_FILTER_SIZE = 3
N_CHANNELS = 4

TARGET_FRAMES = TARGET_FRAMES_MEL
NUM_MEL_BINS = NUM_MEL_BINS_MEL

end_of_conv1_s1 = (TARGET_FRAMES - CONV_FILTER_SIZE + 1) // 2
end_of_conv2_s1 = (end_of_conv1_s1 - CONV_FILTER_SIZE + 1) // 2
end_of_conv1_s2 = (NUM_MEL_BINS - CONV_FILTER_SIZE + 1) // 2
end_of_conv2_s2 = (end_of_conv1_s2 - CONV_FILTER_SIZE + 1) // 2

model = keras.Sequential([
    keras.layers.Input(shape=(TARGET_FRAMES, NUM_MEL_BINS, 1)),
    keras.layers.Conv2D(N_CHANNELS, (CONV_FILTER_SIZE, CONV_FILTER_SIZE), activation="relu"),
    keras.layers.AveragePooling2D((2, 2)),
    keras.layers.Conv2D(N_CHANNELS, (CONV_FILTER_SIZE, CONV_FILTER_SIZE), activation="relu"),
    keras.layers.AveragePooling2D((2, 2)),
    keras.layers.Reshape((end_of_conv2_s2 * end_of_conv2_s1 * N_CHANNELS,)),
    keras.layers.Dense(8, activation="relu"),
    keras.layers.Dense(num_labels),
])
model.compile(
    optimizer="adam",
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
model.summary()


In [ ]:

init_wandb(MODEL_STEM, config={
    "conv_filter_size": CONV_FILTER_SIZE,
    "n_channels": N_CHANNELS,
})

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    callbacks=get_callbacks(10,5,BATCH_SIZE)
)
finish_wandb()

In [ ]:
from utils import (
    build_representative_batches,
    export_keras_model_to_int8_tflite,
)

val_specs = build_representative_batches(test_ds, take=100)

try:
    export_keras_model_to_int8_tflite(model, val_specs, OUT_TFLITE)
    print(f"Success! Wrote {OUT_TFLITE}")
except Exception as e:
    print(f"Conversion failed: {e}")

In [ ]:
from utils import (
    evaluate_tflite_model,
    benchmark_preprocessing,
    create_log_mel_spectrogram,
    fix_audio_length_mel,
    make_audio_datasets,
    FRAME_LENGTH,
    FRAME_STEP,
    FFT_LENGTH_MEL,
    LOWER_EDGE_HERTZ,
    UPPER_EDGE_HERTZ,
    SAMPLE_RATE,
    TARGET_AUDIO_LEN_MEL,
)

# Benchmark preprocessing (STFT + mel projection + log) on raw audio.
_, _, raw_test_ds, _ = make_audio_datasets(batch_size=1)

def _preprocess(audio):
    return create_log_mel_spectrogram(fix_audio_length_mel(audio))

preprocessing_ms = benchmark_preprocessing(_preprocess, raw_test_ds, take=50, warmup=5)
print(f"Avg preprocessing: {preprocessing_ms:.3f} ms")

hyperparams = {
    "conv_filter_size": CONV_FILTER_SIZE,
    "n_channels": N_CHANNELS,
    "target_frames": TARGET_FRAMES,
    "num_mel_bins": NUM_MEL_BINS,
    "frame_length": FRAME_LENGTH,
    "frame_step": FRAME_STEP,
    "fft_length": FFT_LENGTH_MEL,
    "lower_edge_hz": LOWER_EDGE_HERTZ,
    "upper_edge_hz": UPPER_EDGE_HERTZ,
    "target_audio_len": TARGET_AUDIO_LEN_MEL,
    "sample_rate": SAMPLE_RATE,
    "dense_hidden": 8,
    "batch_size": BATCH_SIZE,
}

train_m, test_m, avg_ms = evaluate_tflite_model(
    OUT_TFLITE, MODEL_STEM, train_ds, test_ds,
    hyperparams=hyperparams,
    preprocessing_ms=preprocessing_ms,
)
print(f"Avg inference: {avg_ms:.3f} ms")

In [ ]:
# Reference scores on the 4 sincnet_multi clips, same mel pipeline as gen_mel_input.py.
import re
from pathlib import Path
import numpy as np
import tensorflow as tf
from utils import create_log_mel_spectrogram

REPO = Path("..").resolve()
RS_PATH = REPO / "audio_samples" / "sincnet_multi_tf.rs"
TFLITE_PATH = REPO / "models" / "cnn_mel_tf.tflite"

text = RS_PATH.read_text()
m = re.search(r"scale=([0-9.eE+-]+),\s*zero_point=(-?\d+)", text)
in_scale, in_zp = float(m.group(1)), int(m.group(2))
labels = re.findall(r'expected_label:\s*"([^"]*)"', text)
clips = re.findall(r"CLIP_(\d+): &\[i8\] = &\[(.*?)\];", text, re.S)

interp = tf.lite.Interpreter(model_path=str(TFLITE_PATH))
interp.allocate_tensors()
inp = interp.get_input_details()[0]
outp = interp.get_output_details()[0]
in_q_scale, in_q_zp = inp["quantization"]
out_q_scale, out_q_zp = outp["quantization"]
print(f"cnn_mel input quant: scale={in_q_scale:g}, zp={in_q_zp}, shape={inp['shape'].tolist()}")
print(f"cnn_mel output quant: scale={out_q_scale:g}, zp={out_q_zp}\n")

print(f"{'clip':<5} {'expected':<10} {'predicted':<10} {'score0':>10} {'score1':>10}")
for (n, body), label in zip(clips, labels):
    raw_i8 = np.array([int(v) for v in body.split(",") if v.strip()], np.int8)
    audio = (raw_i8.astype(np.float32) - in_zp) * in_scale
    mel = create_log_mel_spectrogram(tf.constant(audio, dtype=tf.float32)).numpy()
    q = np.clip(np.round(mel / in_q_scale) + in_q_zp, -128, 127).astype(np.int8)
    q = q.reshape(inp["shape"])
    interp.set_tensor(inp["index"], q)
    interp.invoke()
    out_i8 = interp.get_tensor(outp["index"]).astype(np.int32).flatten()
    scores = (out_i8 - out_q_zp) * out_q_scale
    pred = "target" if scores[1] > scores[0] else "non_target"
    print(f"{n:<5} {label:<10} {pred:<10} {scores[0]:>10.4f} {scores[1]:>10.4f}")
